# 09 — Testing & Deployment

Knowing how to test and deploy Node.js applications signals production readiness to interviewers.

---

## Table of Contents
1. Testing Pyramid
2. Unit Testing with Jest
3. Integration Testing
4. API Testing with Supertest
5. Mocking & Stubbing
6. Test-Driven Development (TDD)
7. Docker for Node.js
8. CI/CD Pipeline
9. PM2 & Process Management
10. Environment Management
11. Interview Questions

---
## 1. Testing Pyramid

```
        /\         E2E Tests (few, slow, expensive)
       /  \        Browser automation, Cypress, Playwright
      /    \
     /------\      Integration Tests (some)
    / API,DB  \    Test components working together
   /          \
  /────────────\   Unit Tests (many, fast, cheap)
 / Functions,    \  Test individual functions/classes
/________________\
```

### Testing frameworks for Node.js:
| Tool | Purpose |
|------|--------|
| **Jest** | Unit & integration testing (most popular) |
| **Mocha** | Flexible test runner + Chai (assertions) |
| **Supertest** | HTTP assertion library (API testing) |
| **Sinon** | Mocking, stubbing, spying |
| **Cypress/Playwright** | E2E browser testing |
| **Istanbul/c8** | Code coverage |

---
## 2. Unit Testing with Jest

In [ ]:
// ===== Code to test (calculator.js) =====
function add(a, b) {
    if (typeof a !== 'number' || typeof b !== 'number') {
        throw new TypeError('Arguments must be numbers');
    }
    return a + b;
}

function divide(a, b) {
    if (b === 0) throw new Error('Division by zero');
    return a / b;
}

async function fetchAndAdd(fetchFn, a) {
    const b = await fetchFn();
    return add(a, b);
}

// module.exports = { add, divide, fetchAndAdd };
console.log('Functions defined for testing examples');

In [ ]:
// ===== Jest test examples (calculator.test.js) =====
// This shows the SYNTAX — run with: npx jest

const testExamples = `
const { add, divide, fetchAndAdd } = require('./calculator');

// Grouping with describe
describe('Calculator', () => {

    describe('add()', () => {
        test('adds two positive numbers', () => {
            expect(add(2, 3)).toBe(5);
        });

        test('adds negative numbers', () => {
            expect(add(-1, -1)).toBe(-2);
        });

        test('throws on non-number input', () => {
            expect(() => add('a', 1)).toThrow(TypeError);
            expect(() => add('a', 1)).toThrow('Arguments must be numbers');
        });
    });

    describe('divide()', () => {
        test('divides correctly', () => {
            expect(divide(10, 2)).toBe(5);
        });

        test('throws on division by zero', () => {
            expect(() => divide(10, 0)).toThrow('Division by zero');
        });

        test('returns float for non-even division', () => {
            expect(divide(10, 3)).toBeCloseTo(3.333, 2);
        });
    });

    // Testing async functions
    describe('fetchAndAdd()', () => {
        test('fetches value and adds', async () => {
            const mockFetch = jest.fn().mockResolvedValue(10);
            const result = await fetchAndAdd(mockFetch, 5);
            expect(result).toBe(15);
            expect(mockFetch).toHaveBeenCalledTimes(1);
        });

        test('handles fetch failure', async () => {
            const mockFetch = jest.fn().mockRejectedValue(new Error('Network'));
            await expect(fetchAndAdd(mockFetch, 5)).rejects.toThrow('Network');
        });
    });
});
`;

console.log(testExamples);

### Key Jest matchers to know:
```javascript
// Equality
expect(value).toBe(exact);            // Strict equality (===)
expect(value).toEqual(deep);          // Deep equality (objects/arrays)
expect(value).toStrictEqual(deep);    // Deep + no undefined properties

// Truthiness
expect(value).toBeTruthy();
expect(value).toBeFalsy();
expect(value).toBeNull();
expect(value).toBeUndefined();
expect(value).toBeDefined();

// Numbers
expect(value).toBeGreaterThan(3);
expect(value).toBeLessThanOrEqual(5);
expect(value).toBeCloseTo(0.3, 5);    // For floating point

// Strings
expect(str).toMatch(/regex/);
expect(str).toContain('substring');

// Arrays
expect(arr).toContain(item);
expect(arr).toHaveLength(3);

// Objects
expect(obj).toHaveProperty('key');
expect(obj).toHaveProperty('key', 'value');
expect(obj).toMatchObject({ partial: 'match' });

// Exceptions
expect(() => fn()).toThrow();
expect(() => fn()).toThrow(Error);
expect(() => fn()).toThrow('message');
```

---
## 3. Integration Testing

Integration tests verify that components work together correctly — e.g., your API routes with database operations.

```javascript
// Using an in-memory MongoDB for tests
const { MongoMemoryServer } = require('mongodb-memory-server');

let mongoServer;

beforeAll(async () => {
    mongoServer = await MongoMemoryServer.create();
    await mongoose.connect(mongoServer.getUri());
});

afterAll(async () => {
    await mongoose.disconnect();
    await mongoServer.stop();
});

afterEach(async () => {
    await User.deleteMany({});  // Clean up between tests
});

test('creates and retrieves a user', async () => {
    const user = await User.create({ name: 'Alice', email: 'alice@test.com' });
    const found = await User.findById(user._id);
    expect(found.name).toBe('Alice');
});
```

---
## 4. API Testing with Supertest

Supertest lets you test HTTP endpoints without starting the server.

```javascript
const request = require('supertest');
const app = require('./app'); // Your Express app (don't call .listen()!)

describe('GET /api/users', () => {
    test('returns 200 and list of users', async () => {
        const res = await request(app)
            .get('/api/users')
            .expect('Content-Type', /json/)
            .expect(200);

        expect(res.body).toBeInstanceOf(Array);
        expect(res.body.length).toBeGreaterThan(0);
    });

    test('returns 404 for non-existent user', async () => {
        await request(app)
            .get('/api/users/nonexistent-id')
            .expect(404);
    });
});

describe('POST /api/users', () => {
    test('creates user with valid data', async () => {
        const res = await request(app)
            .post('/api/users')
            .send({ name: 'Bob', email: 'bob@test.com' })
            .expect(201);

        expect(res.body).toHaveProperty('id');
        expect(res.body.name).toBe('Bob');
    });

    test('returns 400 for invalid data', async () => {
        const res = await request(app)
            .post('/api/users')
            .send({ name: '' }) // Missing required fields
            .expect(400);

        expect(res.body).toHaveProperty('error');
    });
});
```

---
## 5. Mocking & Stubbing

### Why mock?
- Isolate the unit under test
- Avoid hitting real databases, APIs, file system
- Control return values and simulate failures
- Speed up tests

```javascript
// Jest mocking

// 1. Mock a function
const mockFn = jest.fn();
mockFn.mockReturnValue(42);
mockFn.mockResolvedValue({ id: 1 });  // For async
mockFn.mockRejectedValue(new Error('fail'));

// 2. Mock a module
jest.mock('./database');  // Auto-mocks all exports
const db = require('./database');
db.findUser.mockResolvedValue({ id: 1, name: 'Alice' });

// 3. Spy on existing methods
const spy = jest.spyOn(console, 'log');
// ... run code ...
expect(spy).toHaveBeenCalledWith('expected message');
spy.mockRestore();

// 4. Mock timers
jest.useFakeTimers();
setTimeout(() => callback(), 1000);
jest.advanceTimersByTime(1000);
expect(callback).toHaveBeenCalled();
jest.useRealTimers();
```

---
## 6. Test-Driven Development (TDD)

### The Red-Green-Refactor cycle:
1. **Red** — Write a failing test first
2. **Green** — Write minimal code to make it pass
3. **Refactor** — Clean up without changing behavior

### Benefits:
- Forces you to think about the API before implementation
- Produces testable, modular code
- Acts as living documentation
- Reduces bugs and regression

### Interview perspective:
> You don't need to practice strict TDD every day, but understanding the concept and being able to demonstrate it in an interview shows discipline and code quality awareness.

---
## 7. Docker for Node.js

### Production Dockerfile:
```dockerfile
# Build stage
FROM node:20-alpine AS builder
WORKDIR /app
COPY package*.json ./
RUN npm ci --only=production

# Production stage
FROM node:20-alpine
WORKDIR /app

# Create non-root user
RUN addgroup -S appgroup && adduser -S appuser -G appgroup

COPY --from=builder /app/node_modules ./node_modules
COPY . .

USER appuser
EXPOSE 3000
ENV NODE_ENV=production

HEALTHCHECK --interval=30s --timeout=3s \
    CMD wget --no-verbose --tries=1 --spider http://localhost:3000/health || exit 1

CMD ["node", "index.js"]
```

### Key Docker practices:
- Use `alpine` images (smaller, more secure)
- Multi-stage builds (smaller final image)
- Run as non-root user
- Use `.dockerignore` (exclude `node_modules`, `.git`, `*.md`)
- Use `npm ci` (not `npm install`) for deterministic builds
- Set `NODE_ENV=production`
- Add `HEALTHCHECK`

### docker-compose.yml:
```yaml
version: '3.8'
services:
  app:
    build: .
    ports: ['3000:3000']
    environment:
      - NODE_ENV=production
      - DB_URL=mongodb://mongo:27017/myapp
    depends_on: [mongo, redis]

  mongo:
    image: mongo:7
    volumes: ['mongo-data:/data/db']

  redis:
    image: redis:7-alpine

volumes:
  mongo-data:
```

---
## 8. CI/CD Pipeline

### GitHub Actions example:
```yaml
# .github/workflows/ci.yml
name: CI
on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        node-version: [18, 20, 22]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: ${{ matrix.node-version }}
          cache: 'npm'
      - run: npm ci
      - run: npm test
      - run: npm run lint

  deploy:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t myapp .
      - run: docker push myregistry/myapp:latest
```

### CI/CD best practices:
- Run tests on every push/PR
- Test on multiple Node.js versions
- Cache `node_modules` for speed
- Lint before merge
- Auto-deploy on main branch merge
- Use environment-specific configs

---
## 9. PM2 & Process Management

```bash
# Basic PM2 commands
pm2 start app.js              # Start app
pm2 start app.js -i max       # Cluster mode (all cores)
pm2 start app.js --name myapp # Named process
pm2 list                      # List processes
pm2 monit                     # Monitor dashboard
pm2 logs                      # View logs
pm2 reload myapp              # Zero-downtime restart
pm2 stop myapp                # Stop
pm2 delete myapp              # Remove
pm2 save                      # Save process list
pm2 startup                   # Generate startup script
```

### ecosystem.config.js:
```javascript
module.exports = {
    apps: [{
        name: 'myapp',
        script: 'index.js',
        instances: 'max',       // Cluster mode
        exec_mode: 'cluster',
        env: {
            NODE_ENV: 'development',
            PORT: 3000
        },
        env_production: {
            NODE_ENV: 'production',
            PORT: 8080
        },
        max_memory_restart: '300M',
        error_file: './logs/err.log',
        out_file: './logs/out.log',
    }]
};
```

---
## 10. Environment Management

```javascript
// Using dotenv
require('dotenv').config();

// .env file (NEVER commit this!):
// DB_URL=mongodb://localhost:27017/myapp
// JWT_SECRET=super-secret-key
// PORT=3000

// .env.example (commit this — template for other devs):
// DB_URL=
// JWT_SECRET=
// PORT=3000

// config.js — centralized config
const config = {
    port: process.env.PORT || 3000,
    db: {
        url: process.env.DB_URL || 'mongodb://localhost:27017/dev',
    },
    jwt: {
        secret: process.env.JWT_SECRET,
        expiresIn: '7d',
    },
    isProduction: process.env.NODE_ENV === 'production',
};

// Validate required env vars at startup
if (!config.jwt.secret) {
    throw new Error('JWT_SECRET is required');
}
```

### .gitignore essentials:
```
node_modules/
.env
.env.local
*.log
dist/
coverage/
```

---
## 11. Interview Questions & Answers

### Q1: What types of tests should a Node.js application have?
**A:** Unit tests (individual functions, most numerous), integration tests (components working together, database queries, API routes), and E2E tests (full user workflows, fewest). Follow the testing pyramid — many unit tests, fewer integration tests, few E2E tests.

### Q2: How do you test an Express API?
**A:** Use Supertest to make HTTP requests to the app without starting a server. Export your Express `app` without calling `.listen()`. Test status codes, response bodies, headers. Use an in-memory database for integration tests.

### Q3: What is mocking and when do you use it?
**A:** Mocking replaces real dependencies with controlled substitutes. Use it to isolate units from databases, APIs, file system, and time. Jest provides `jest.fn()`, `jest.mock()`, and `jest.spyOn()`. Mock at boundaries (DB calls, HTTP requests), not internal logic.

### Q4: How do you deploy a Node.js application?
**A:** Containerize with Docker using multi-stage builds. Use PM2 for process management (cluster mode, auto-restart). Set up CI/CD (GitHub Actions, Jenkins) for automated testing and deployment. Use environment variables for config. Deploy to cloud (AWS ECS, GCP Cloud Run, Heroku, etc.).

### Q5: What's the difference between `npm install` and `npm ci`?
**A:** `npm install` reads `package.json`, may update `package-lock.json`, and installs to existing `node_modules`. `npm ci` reads `package-lock.json` exactly, deletes and recreates `node_modules`, and never modifies the lock file. Use `npm ci` in CI/CD for reproducible builds.

### Q6: How do you handle environment-specific configuration?
**A:** Use `.env` files with `dotenv` for local development, environment variables in production (injected by Docker/Kubernetes/cloud platform). Create a centralized config module that validates required variables at startup. Never commit secrets to version control.